# Multimodal Emotion Detection
This notebook demonstrates using a unified model for emotion detection in both text and images.

In [3]:
# Import required libraries
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import img_to_array
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd
import seaborn as sns
from transformers import pipeline
import warnings
warnings.filterwarnings('ignore')

c:\Users\Shubhanshu\AppData\Local\Programs\Python\Python38\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]


In [4]:
# Load the multimodal model
model_path = 'MULTIMODAL_detection_model_hf.h5'
multimodal_model = load_model(model_path)

# Load face classifier for image preprocessing
face_classifier = cv2.CascadeClassifier('/../haarcascade_frontalface_default.xml')

# Emotion labels
emotion_labels = ['Angry', 'Disgust', 'Fear', 'Happy', 'Neutral', 'Sad', 'Surprise']

## Model Summary

In [5]:
# Display model architecture
print("Multimodal Emotion Detection Model Summary:")
print("="*50)
multimodal_model.summary()

Multimodal Emotion Detection Model Summary:
Model: "sequential_5"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_22 (Conv2D)          (None, 46, 46, 64)        640       
                                                                 
 max_pooling2d_14 (MaxPooli  (None, 23, 23, 64)        0         
 ng2D)                                                           
                                                                 
 dropout_21 (Dropout)        (None, 23, 23, 64)        0         
                                                                 
 conv2d_23 (Conv2D)          (None, 21, 21, 256)       147712    
                                                                 
 max_pooling2d_15 (MaxPooli  (None, 10, 10, 256)       0         
 ng2D)                                                           
                                                                 
 flatten_5

In [14]:
# Visualize model architecture
tf.keras.utils.plot_model(
    multimodal_model, 
    to_file='model_architecture.png', 
    show_shapes=True, 
    show_layer_names=True,
    rankdir='TB'
)

# Display the architecture image
img = Image.open('model_architecture.png')
plt.figure(figsize=(12, 8))
plt.imshow(img)
plt.axis('off')
plt.title('Multimodal Model Architecture')
plt.show()

You must install pydot (`pip install pydot`) and install graphviz (see instructions at https://graphviz.gitlab.io/download/) for plot_model to work.


UnidentifiedImageError: cannot identify image file 'C:\\Users\\Shubhanshu\\college\\Major\\Emotion_Detection_CNN\\multimodal\\model_architecture.png'

## Text Emotion Detection

In [ ]:
def preprocess_text(text):
    """Preprocess text for the multimodal model"""
    # Convert text to numerical representation
    # This would depend on how the model was trained
    # For demonstration, using a simple tokenization approach
    from tensorflow.keras.preprocessing.text import Tokenizer
    from tensorflow.keras.preprocessing.sequence import pad_sequences
    
    tokenizer = Tokenizer(num_words=10000)
    tokenizer.fit_on_texts([text])
    sequences = tokenizer.texts_to_sequences([text])
    return pad_sequences(sequences, maxlen=100)

def predict_text_emotion(text):
    """Predict emotion from text using the multimodal model"""
    try:
        # Preprocess text
        processed_text = preprocess_text(text)
        
        # For multimodal model, we might need to create dummy image input
        # or use text-only branch if available
        dummy_image = np.zeros((1, 48, 48, 1))  # Dummy image input
        
        # Predict using the model
        prediction = multimodal_model.predict([processed_text, dummy_image])[0]
        
        # Create emotion dictionary
        emotions = {emotion_labels[i]: prediction[i] for i in range(len(emotion_labels))}
        return emotions
    except Exception as e:
        print(f"Error in text prediction: {e}")
        # Fallback to Hugging Face pipeline
        classifier = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base")
        result = classifier(text)
        return {item['label']: item['score'] for item in result}

# Test text emotion detection
sample_texts = [
    "I am so happy today!",
    "This makes me really angry.",
    "I feel sad and lonely.",
    "What a wonderful surprise!"
]

print("Text Emotion Detection Results:")
print("="*40)

for text in sample_texts:
    emotions = predict_text_emotion(text)
    dominant_emotion = max(emotions, key=emotions.get)
    confidence = emotions[dominant_emotion]
    
    print(f"\nText: '{text}'")
    print(f"Dominant Emotion: {dominant_emotion} ({confidence:.3f})")
    
    # Create visualization
    plt.figure(figsize=(10, 4))
    emotions_df = pd.DataFrame(list(emotions.items()), columns=['Emotion', 'Probability'])
    emotions_df = emotions_df.sort_values('Probability', ascending=True)
    
    plt.barh(emotions_df['Emotion'], emotions_df['Probability'])
    plt.xlabel('Probability')
    plt.title(f"Emotion Distribution for: '{text}'")
    plt.tight_layout()
    plt.show()

Text Emotion Detection Results:
Error in text prediction: name 'multimodal_model' is not defined


NameError: name 'pipeline' is not defined

## Image Emotion Detection

In [ ]:
def preprocess_image(image_path):
    """Preprocess image for emotion detection"""
    image = cv2.imread(image_path)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    faces = face_classifier.detectMultiScale(gray, 1.3, 5)
    
    if len(faces) == 0:
        return None, None, image
    
    face_images = []
    face_coords = []
    
    for (x, y, w, h) in faces:
        roi_gray = gray[y:y+h, x:x+w]
        roi_gray = cv2.resize(roi_gray, (48, 48), interpolation=cv2.INTER_AREA)
        roi = roi_gray.astype('float') / 255.0
        roi = img_to_array(roi)
        roi = np.expand_dims(roi, axis=0)
        
        face_images.append(roi)
        face_coords.append((x, y, w, h))
    
    return face_images, face_coords, image

def predict_image_emotion(image_path):
    """Predict emotion from image using the multimodal model"""
    face_images, face_coords, original_image = preprocess_image(image_path)
    
    if face_images is None:
        return None, original_image
    
    results = []
    annotated_image = original_image.copy()
    
    for i, (face_img, (x, y, w, h)) in enumerate(zip(face_images, face_coords)):
        # For multimodal model, create dummy text input
        dummy_text = np.zeros((1, 100))  # Dummy text input
        
        # Predict emotion
        prediction = multimodal_model.predict([dummy_text, face_img])[0]
        emotions = {emotion_labels[j]: prediction[j] for j in range(len(emotion_labels))}
        results.append(emotions)
        
        # Annotate image
        dominant_emotion = max(emotions, key=emotions.get)
        confidence = emotions[dominant_emotion]
        
        cv2.rectangle(annotated_image, (x, y), (x+w, y+h), (0, 255, 255), 2)
        cv2.putText(annotated_image, f'{dominant_emotion}: {confidence:.2f}', 
                   (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
    
    return results, annotated_image

# Test with sample image (you can replace with your own image path)
# For demonstration, let's create a simple test
print("Image Emotion Detection:")
print("="*30)
print("Please provide an image path to test image emotion detection.")
print("Example usage:")
print("image_path = 'path/to/your/image.jpg'")
print("results, annotated_img = predict_image_emotion(image_path)")

In [ ]:
# Example usage for image emotion detection
# Uncomment and modify the path below to test with your own image

# image_path = 'path/to/your/image.jpg'
# results, annotated_image = predict_image_emotion(image_path)

# if results:
#     # Display original and annotated images
#     fig, axes = plt.subplots(1, 2, figsize=(15, 6))
#     
#     # Original image
#     original = cv2.imread(image_path)
#     original_rgb = cv2.cvtColor(original, cv2.COLOR_BGR2RGB)
#     axes[0].imshow(original_rgb)
#     axes[0].set_title('Original Image')
#     axes[0].axis('off')
#     
#     # Annotated image
#     annotated_rgb = cv2.cvtColor(annotated_image, cv2.COLOR_BGR2RGB)
#     axes[1].imshow(annotated_rgb)
#     axes[1].set_title('Emotion Detection Results')
#     axes[1].axis('off')
#     
#     plt.tight_layout()
#     plt.show()
#     
#     # Display emotion probabilities for each face
#     for i, emotions in enumerate(results):
#         print(f"\nFace {i+1} Emotions:")
#         emotions_df = pd.DataFrame(list(emotions.items()), columns=['Emotion', 'Probability'])
#         emotions_df = emotions_df.sort_values('Probability', ascending=False)
#         
#         plt.figure(figsize=(10, 4))
#         plt.bar(emotions_df['Emotion'], emotions_df['Probability'])
#         plt.xlabel('Emotion')
#         plt.ylabel('Probability')
#         plt.title(f'Emotion Distribution for Face {i+1}')
#         plt.xticks(rotation=45)
#         plt.tight_layout()
#         plt.show()
# else:
#     print("No faces detected in the image.")

## Interactive Demo

In [ ]:
# Interactive text emotion detection
def interactive_text_demo():
    print("Interactive Text Emotion Detection")
    print("Type 'quit' to exit")
    
    while True:
        text = input("\nEnter text to analyze: ")
        if text.lower() == 'quit':
            break
        
        emotions = predict_text_emotion(text)
        dominant_emotion = max(emotions, key=emotions.get)
        confidence = emotions[dominant_emotion]
        
        print(f"Dominant Emotion: {dominant_emotion} ({confidence:.3f})")
        print("All emotions:")
        for emotion, prob in sorted(emotions.items(), key=lambda x: x[1], reverse=True):
            print(f"  {emotion}: {prob:.3f}")

# Uncomment to run interactive demo
# interactive_text_demo()

## Model Performance Analysis

In [ ]:
# Analyze model layers and parameters
print("Model Analysis:")
print("="*20)
print(f"Total parameters: {multimodal_model.count_params():,}")
print(f"Number of layers: {len(multimodal_model.layers)}")

# Layer information
print("\nLayer Details:")
for i, layer in enumerate(multimodal_model.layers):
    print(f"{i+1:2d}. {layer.name:20s} - {layer.__class__.__name__:15s} - Output: {layer.output_shape}")

# Model configuration
print("\nModel Configuration:")
config = multimodal_model.get_config()
print(f"Model name: {config.get('name', 'N/A')}")
print(f"Number of inputs: {len(multimodal_model.inputs)}")
print(f"Number of outputs: {len(multimodal_model.outputs)}")

for i, input_layer in enumerate(multimodal_model.inputs):
    print(f"Input {i+1} shape: {input_layer.shape}")

for i, output_layer in enumerate(multimodal_model.outputs):
    print(f"Output {i+1} shape: {output_layer.shape}")

## Conclusion

This notebook demonstrates the usage of a multimodal emotion detection model that can process both text and image inputs. The model provides emotion predictions across 7 categories: Angry, Disgust, Fear, Happy, Neutral, Sad, and Surprise.

Key features:
- Unified model for both text and image emotion detection
- Real-time face detection and emotion analysis
- Comprehensive model architecture visualization
- Interactive testing capabilities

The multimodal approach allows for more robust emotion detection by leveraging both textual and visual cues.